# 🔬 Esophagitis vs Normal Z-Line — CNN Classifier

**Task:** Binary image classification from endoscopy images  
**Classes:** `esophagitis` (Grade A–D inflammation) vs `normal-z-line` (healthy)  
**Dataset:** [The Kvasir Dataset](https://www.kaggle.com/datasets/meetnagadia/kvasir-dataset) — use the `esophagitis` and `normal-z-line` folders  

**Two approaches included:**
1. Custom CNN from scratch (educational, good baseline)
2. Transfer Learning with EfficientNetB0 (recommended for small medical datasets)

---

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 2. Configuration

In [ ]:
# ── Dataset path ──────────────────────────────────────────────────────────────
# Kvasir dataset structure expected:
#   /kaggle/input/kvasir-dataset/kvasir-dataset/esophagitis/   (*.jpg)
#   /kaggle/input/kvasir-dataset/kvasir-dataset/normal-z-line/ (*.jpg)

DATA_ROOT = Path('/kaggle/input/kvasir-dataset/kvasir-dataset')

# ── Hyper-parameters ──────────────────────────────────────────────────────────
IMG_SIZE   = 224        # pixels (both height and width)
BATCH_SIZE = 32
EPOCHS_CNN = 40         # custom CNN
EPOCHS_TL  = 20         # transfer learning fine-tune
LR         = 1e-3
DROPOUT    = 0.5

# Class names (folder names)
CLASSES    = ['esophagitis', 'normal-z-line']
CLASS_LABELS = {c: i for i, c in enumerate(CLASSES)}
print('Class mapping:', CLASS_LABELS)

## 3. Exploratory Data Analysis

In [ ]:
# Count images per class
counts = {}
for cls in CLASSES:
    p = DATA_ROOT / cls
    imgs = list(p.glob('*.jpg')) + list(p.glob('*.png'))
    counts[cls] = len(imgs)
    print(f'{cls:20s}: {len(imgs):4d} images')

print(f'\nTotal: {sum(counts.values())} images')

# Class balance bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(counts.keys(), counts.values(), color=['#E85D30', '#2A7BBE'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of images')
for i, (cls, n) in enumerate(counts.items()):
    axes[0].text(i, n + 5, str(n), ha='center', fontweight='bold')

# Sample images
axes[1].axis('off')
axes[1].set_title('Sample grid (see next cell)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import load_img

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample endoscopy images', fontsize=14, fontweight='bold')

for row, cls in enumerate(CLASSES):
    imgs = sorted((DATA_ROOT / cls).glob('*.jpg'))[:5]
    for col, path in enumerate(imgs):
        img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Data Pipeline with Augmentation

In [ ]:
# ── Data generators ───────────────────────────────────────────────────────────
# Medical image augmentation: conservative — avoid flips that change clinical meaning

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest',
)

val_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2,
)

train_gen = train_datagen.flow_from_directory(
    DATA_ROOT,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    classes=CLASSES,
    class_mode='binary',
    subset='training',
    seed=SEED,
    shuffle=True,
)

val_gen = val_datagen.flow_from_directory(
    DATA_ROOT,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    classes=CLASSES,
    class_mode='binary',
    subset='validation',
    seed=SEED,
    shuffle=False,
)

print(f'Training samples  : {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Class indices     : {train_gen.class_indices}')

In [ ]:
# Visualise augmented samples
batch_imgs, batch_labels = next(train_gen)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Augmented training samples', fontsize=13, fontweight='bold')
label_names = {v: k for k, v in train_gen.class_indices.items()}

for i, ax in enumerate(axes.flat):
    ax.imshow(batch_imgs[i])
    ax.set_title(label_names[int(batch_labels[i])], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Custom CNN Model

In [ ]:
def conv_block(x, filters, dropout_rate=0.25):
    """Conv → BN → ReLU → Conv → BN → ReLU → MaxPool → Dropout"""
    x = layers.Conv2D(filters, (3, 3), padding='same', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(dropout_rate)(x)
    return x


def build_custom_cnn(input_shape=(224, 224, 3), dropout=0.5):
    inputs = keras.Input(shape=input_shape, name='input')

    x = conv_block(inputs, 32,  dropout_rate=0.2)   # 224 → 112
    x = conv_block(x,      64,  dropout_rate=0.25)  # 112 → 56
    x = conv_block(x,      128, dropout_rate=0.3)   # 56  → 28
    x = conv_block(x,      256, dropout_rate=0.3)   # 28  → 14

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='output')(x)

    return keras.Model(inputs, outputs, name='EsophagitisNet')


cnn_model = build_custom_cnn(dropout=DROPOUT)
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

# ── Callbacks ────────────────────────────────────────────────────────────────
cnn_callbacks = [
    callbacks.ModelCheckpoint(
        'best_cnn.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor='val_auc',
        patience=10,
        mode='max',
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
]

In [ ]:
history_cnn = cnn_model.fit(
    train_gen,
    epochs=EPOCHS_CNN,
    validation_data=val_gen,
    callbacks=cnn_callbacks,
    verbose=1,
)

## 6. Transfer Learning — EfficientNetB0

In [ ]:
# EfficientNetB0 expects [0, 255] input — don't rescale!
train_datagen_tl = ImageDataGenerator(
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest',
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
)

val_datagen_tl = ImageDataGenerator(
    validation_split=0.2,
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
)

train_gen_tl = train_datagen_tl.flow_from_directory(
    DATA_ROOT, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, classes=CLASSES,
    class_mode='binary', subset='training', seed=SEED, shuffle=True,
)

val_gen_tl = val_datagen_tl.flow_from_directory(
    DATA_ROOT, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, classes=CLASSES,
    class_mode='binary', subset='validation', seed=SEED, shuffle=False,
)

In [ ]:
def build_efficientnet(dropout=0.5):
    base = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = False  # freeze backbone initially

    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return keras.Model(inputs, outputs, name='EfficientNet_Esophagitis'), base


tl_model, base_model = build_efficientnet(dropout=DROPOUT)

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Recall(name='recall')]
)

tl_callbacks = [
    callbacks.ModelCheckpoint('best_tl.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_auc', patience=8, mode='max', restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
]

# Phase 1: train the head only
print('Phase 1 — training head only (backbone frozen)...')
history_tl_head = tl_model.fit(
    train_gen_tl, epochs=10,
    validation_data=val_gen_tl,
    callbacks=tl_callbacks, verbose=1,
)

In [ ]:
# Phase 2: fine-tune last 30 layers of backbone
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),   # lower LR for fine-tuning
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Recall(name='recall')]
)

print(f'Trainable layers: {sum(1 for l in tl_model.layers if l.trainable)}')
print('Phase 2 — fine-tuning backbone (last 30 layers)...')

history_tl_finetune = tl_model.fit(
    train_gen_tl, epochs=EPOCHS_TL,
    validation_data=val_gen_tl,
    callbacks=tl_callbacks, verbose=1,
)

## 7. Training History Plots

In [ ]:
def plot_history(history, title='Training history'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    metrics = [
        ('loss',     'Loss',     'lower is better'),
        ('accuracy', 'Accuracy', 'higher is better'),
        ('auc',      'AUC',      'higher is better'),
    ]

    for ax, (key, label, note) in zip(axes, metrics):
        ax.plot(history.history[key],     label='train', linewidth=2)
        ax.plot(history.history[f'val_{key}'], label='val', linewidth=2, linestyle='--')
        ax.set_title(f'{label} ({note})', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_cnn, title='Custom CNN — training history')

## 8. Evaluation & Metrics

In [ ]:
def evaluate_model(model, gen, model_name='Model'):
    """Comprehensive evaluation: accuracy, AUC, confusion matrix, ROC curve."""
    gen.reset()
    y_prob = model.predict(gen, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = gen.classes

    acc  = accuracy_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)

    print(f'\n──── {model_name} ────')
    print(f'Accuracy : {acc:.4f}')
    print(f'ROC-AUC  : {auc:.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(model_name, fontsize=13, fontweight='bold')

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0]
    )
    axes[0].set_title('Confusion matrix')
    axes[0].set_ylabel('True label')
    axes[0].set_xlabel('Predicted label')

    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc:.3f}')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[1].set_xlabel('False positive rate')
    axes[1].set_ylabel('True positive rate')
    axes[1].set_title('ROC curve')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    return {'accuracy': acc, 'auc': auc, 'y_true': y_true, 'y_prob': y_prob}


cnn_results = evaluate_model(cnn_model, val_gen, 'Custom CNN')

In [ ]:
tl_results = evaluate_model(tl_model, val_gen_tl, 'EfficientNetB0 (fine-tuned)')

## 9. Grad-CAM Visualisation

In [ ]:
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Compute Grad-CAM heatmap for a single image."""
    grad_model = keras.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def show_gradcam(model, img_path, last_conv, label_name, preprocess_fn=None):
    from tensorflow.keras.preprocessing.image import load_img, img_to_array
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = img_to_array(img)
    if preprocess_fn:
        inp = preprocess_fn(arr.copy()[np.newaxis])
    else:
        inp = arr.copy()[np.newaxis] / 255.0

    heatmap = make_gradcam_heatmap(inp, model, last_conv)

    # Overlay
    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(np.uint8(arr), 0.6, heatmap_colored, 0.4, 0)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(arr.astype('uint8'))
    axes[0].set_title(f'Original — {label_name}')
    axes[0].axis('off')
    axes[1].imshow(heatmap_resized, cmap='jet')
    axes[1].set_title('Grad-CAM heatmap')
    axes[1].axis('off')
    axes[2].imshow(superimposed)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()


# Custom CNN: last conv layer is inside the last conv_block
# Find the last Conv2D layer name:
last_conv = [l.name for l in cnn_model.layers if 'conv2d' in l.name][-1]
print('Last conv layer (custom CNN):', last_conv)

# Show Grad-CAM for one esophagitis and one normal sample
for cls in CLASSES:
    sample_img = next((DATA_ROOT / cls).glob('*.jpg'))
    show_gradcam(cnn_model, sample_img, last_conv, cls, preprocess_fn=None)

## 10. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model'   : ['Custom CNN', 'EfficientNetB0 (fine-tuned)'],
    'Accuracy': [cnn_results['accuracy'], tl_results['accuracy']],
    'AUC'     : [cnn_results['auc'],      tl_results['auc']],
})

comparison = comparison.set_index('Model')
print(comparison.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
comparison.plot(kind='bar', ax=ax, rot=0, color=['#2A7BBE', '#E85D30'], edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.05)
ax.set_title('Model comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

## 11. Save & Submission

```python
# Save the better model
best = tl_model  # or cnn_model
best.save('esophagitis_classifier.keras')

# For Kaggle output
import shutil
shutil.copy('esophagitis_classifier.keras', '/kaggle/working/')
```

---

## Notes & Next Steps

| Topic | Suggestion |
|---|---|
| **Class imbalance** | If classes are unbalanced, add `class_weight` to `model.fit()` |
| **Better augmentation** | Try `albumentations` for CLAHE, elastic deform, stain normalisation |
| **Bigger backbone** | EfficientNetB3/B4 or ConvNeXt for more capacity |
| **Self-supervised pre-training** | Use DINO or SimCLR on unlabelled endoscopy data first |
| **Test-Time Augmentation (TTA)** | Average predictions over 5–10 augmented versions of each test image |
| **Threshold tuning** | Optimise the decision threshold for recall (clinical priority: miss fewer esophagitis cases) |
| **k-fold cross-validation** | With small datasets (~1000 images), 5-fold CV gives more reliable estimates |
